In [1]:
import pandas as pd

# 포인트 사용 기록 테이블

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_pointhistory`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

          id  delta_point                created_at  user_id  \
0  145203381         -500 2023-05-18 12:24:21+00:00  1228792   
1  145439633         -500 2023-05-18 12:37:14+00:00  1273533   
2  145524609         -500 2023-05-18 12:41:47+00:00   880470   
3  145769790         -500 2023-05-18 12:54:49+00:00   863427   
4  145961455        -1000 2023-05-18 13:04:01+00:00  1303729   

   user_question_record_id  
0                 69029243  
1                 70891824  
2                  4028580  
3                 71932526  
4                 56455390  


## 결측치 확인 및 데이터 정보 확인

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2338918 entries, 0 to 2338917
Data columns (total 5 columns):
 #   Column                   Dtype              
---  ------                   -----              
 0   id                       Int64              
 1   delta_point              Int64              
 2   created_at               datetime64[us, UTC]
 3   user_id                  Int64              
 4   user_question_record_id  Int64              
dtypes: Int64(4), datetime64[us, UTC](1)
memory usage: 98.1 MB


In [5]:
df.isna().sum()

id                            0
delta_point                   0
created_at                    0
user_id                       0
user_question_record_id    2992
dtype: int64

* `user_question_record_id` 2992건 결측 발견

In [6]:
df[df['user_question_record_id'].isna()]

,id,delta_point,created_at,user_id,user_question_record_id
1729,331288275,200,2023-06-23 17:47:11+00:00,1203119,<NA>
1733,331693311,1000,2023-06-24 16:07:41+00:00,1402393,<NA>
1736,331844884,500,2023-06-25 06:26:46+00:00,1263081,<NA>
1740,332101446,200,2023-06-26 02:54:37+00:00,1020695,<NA>
1742,332285669,1000,2023-06-26 20:14:01+00:00,878500,<NA>
...,...,...,...,...,...
1175793,340568649,50,2024-02-04 11:16:38+00:00,873406,<NA>
1175798,340609534,60,2024-03-10 03:27:03+00:00,1227249,<NA>
1175799,340617339,50,2024-03-19 12:56:09+00:00,1583358,<NA>
1175801,340639081,50,2024-04-12 10:38:04+00:00,863169,<NA>


In [7]:
# 1. 각각의 집계 결과 계산
na_size = df[df['user_question_record_id'].isna()].groupby('delta_point').size()
total_size = df.groupby('delta_point').size()

# 2. 하나의 데이터프레임으로 나란히 병합
result = pd.concat([na_size, total_size], axis=1, keys=['na_count', 'total_count']).fillna(0)

# 3. 비율(%) 파생변수까지 추가하면 더 직관적입니다
result['na_ratio(%)'] = (result['na_count'] / result['total_count'] * 100).round(2)

result

,na_count,total_count,na_ratio(%)
delta_point,,,
-30,1.0,1,100.0
50,322.0,322,100.0
60,47.0,47,100.0
70,16.0,16,100.0
80,10.0,10,100.0
90,7.0,7,100.0
100,10.0,10,100.0
110,6.0,6,100.0
120,6.0,6,100.0


In [8]:
# 전처리 수행 대상 테이블 호출
temp_sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.events`
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

print(temp_df.head())

   id           title  plus_point event_type  is_expired  \
0   1   코드잇 은행 가입 이벤트         500       FCFS           1   
1   3   예고 영상 기대평 이벤트         500       FCFS           1   
2   2  코드잇 멤버십 가입 이벤트        1000       FCFS           1   

                 created_at  
0 2023-06-20 11:56:38+00:00  
1 2023-09-24 17:05:59+00:00  
2 2023-08-08 07:43:45+00:00  


* 200, 777, 1000에서 발견된 결측은 충전 과정에서 발생된 history로 판단된다.
* 또한 500포인트와 1000포인트를 지급하는 이벤트를 진행했으므로, 500포인트 역시 user_question_record_id 결측 설명이 가능.

* -30, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 210, 220, 230, 240, 250, 260, 270, 280, 300 포인트 확인 필요

* 과거 다른 형태의 요금제가 있었는지 확인

In [12]:
# 전처리 수행 대상 테이블 호출
temp_sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_paymenthistory`
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

print(temp_df.head())

      id  productId phone_type                created_at  user_id
0  89584  heart.777          A 2023-06-06 04:58:49+00:00   835888
1  89585  heart.200          A 2023-06-06 04:59:22+00:00   835888
2   2403  heart.777          A 2023-05-14 04:22:44+00:00   837641
3  79774  heart.777          A 2023-05-29 10:13:55+00:00   837737
4    195  heart.777          A 2023-05-13 23:10:10+00:00   837842


In [13]:
temp_df.groupby(by='productId').count()

,id,phone_type,created_at,user_id
productId,,,,
heart.1000,19309,19309,19309,19309
heart.200,15822,15822,15822,15822
heart.4000,2136,2136,2136,2136
heart.777,57873,57873,57873,57873


In [15]:
print(df['created_at'].min())
print(df['created_at'].max())
print(temp_df['created_at'].min())
print(temp_df['created_at'].max())

2023-04-28 12:27:49+00:00
2024-05-08 01:36:18+00:00
2023-05-13 21:28:34+00:00
2024-05-08 14:12:45+00:00


* 요금제의 변경은 없었던 것으로 판단되며,
* 기간 중 4000포인트의 충전이 꽤 있었으나, 포인트 기록 히스토리에서는 보이지 않고,
* 777포인트도 수의 차이가 보임.

* 기획서 상 포인트 수급처를 추가로 확인해 본 결과 출석체크도 있긴 함.
* 그리고 연속 출석 일 수등 다양한 조건에 따라 보너스 포인트를 주기도 함. (기획서 상 240포인트는 확인)
* (추정) 이번주 출석 일 수 / 이번달 출석 일 수 / 연속 출석 일 수 등 다양한 조합에 따라 포인트가 쌓이는 것이 아닐지 추측
* 또한 주요 포인트 사용처는 초성확인으로 각각 200포인트 , 500포인트, 1000포인트가 소모.
* -300, -10, -30 등에 대한 사용처 추가 확인이 필요해 보임.

## 중복값 확인

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df[['delta_point', 'created_at', 'user_id', 'user_question_record_id']].duplicated().sum()

np.int64(1939)

* id값 제외시 중복 1939건 발견
* 중복 이벤트 가능성 있음 확인 필요.

In [11]:
# 중복 검사 대상 컬럼 목록
cols = ['delta_point', 'created_at', 'user_id', 'user_question_record_id']

# 1. 중복 데이터만 필터링한 뒤 해당 컬럼 기준으로 정렬해서 출력
duplicated_df = df[df[cols].duplicated(keep=False)].sort_values(by=cols)

# 2. 상위 10개 출력 (연달아 붙어있는 중복 데이터 확인)
with pd.option_context('display.max_columns', None):
    display(duplicated_df.head(10)) 

,id,delta_point,created_at,user_id,user_question_record_id
1268983,77956321,5,2023-05-13 16:26:32+00:00,1198097,38953394
1268984,77956327,5,2023-05-13 16:26:32+00:00,1198097,38953394
116799,80320871,5,2023-05-14 01:24:22+00:00,1234279,40107388
1269416,80320874,5,2023-05-14 01:24:22+00:00,1234279,40107388
117496,84225200,5,2023-05-14 05:52:04+00:00,1238029,42011932
1258151,84225197,5,2023-05-14 05:52:04+00:00,1238029,42011932
73055,85089876,5,2023-05-14 06:49:10+00:00,1091180,42433065
73056,85089882,5,2023-05-14 06:49:10+00:00,1091180,42433065
73707,88513469,5,2023-05-14 10:32:05+00:00,1091180,44098497
118386,88513465,5,2023-05-14 10:32:05+00:00,1091180,44098497


* 중복 발생한 데이터로 보는 것이 적합하다고 판단.